## Summary Statistics for graduates_person_level Dataset

Comprehensive overview of the dataset structure and variable distributions.

In [ ]:
# Basic dataset information
print("=== DATASET OVERVIEW ===")
print(f"Dataset shape: {graduates_person_level.shape}")
print(f"Number of observations: {graduates_person_level.shape[0]:,}")
print(f"Number of variables: {graduates_person_level.shape[1]:,}")
print(f"Memory usage: {graduates_person_level.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nData types:")
print(graduates_person_level.dtypes.value_counts())
print("\n" + "="*60)

In [ ]:
# Missing values analysis
print("=== MISSING VALUES ANALYSIS ===")
missing_summary = graduates_person_level.isnull().sum()
missing_percentage = (missing_summary / len(graduates_person_level)) * 100

missing_df = pd.DataFrame({
    'Variable': missing_summary.index,
    'Missing_Count': missing_summary.values,
    'Missing_Percentage': missing_percentage.values
}).sort_values('Missing_Percentage', ascending=False)

print(f"Variables with missing values: {(missing_df['Missing_Count'] > 0).sum()}")
print(f"Variables with no missing values: {(missing_df['Missing_Count'] == 0).sum()}")
print("\nTop 20 variables with highest missing values:")
print(missing_df.head(20).to_string(index=False))
print("\n" + "="*60)

In [ ]:
import numpy as np
# Numeric variables summary statistics
print("=== NUMERIC VARIABLES SUMMARY ===")
numeric_cols = graduates_person_level.select_dtypes(include=[np.number]).columns
print(f"Number of numeric variables: {len(numeric_cols)}")

if len(numeric_cols) > 0:
    numeric_summary = graduates_person_level[numeric_cols].describe()
    print("\nDetailed summary statistics for numeric variables:")
    print(numeric_summary.round(3))
    
    # Additional statistics for numeric variables
    print("\nAdditional numeric statistics:")
    additional_stats = pd.DataFrame({
        'Variable': numeric_cols,
        'Median': graduates_person_level[numeric_cols].median(),
        'Mode': graduates_person_level[numeric_cols].mode().iloc[0] if len(graduates_person_level[numeric_cols].mode()) > 0 else np.nan,
        'Variance': graduates_person_level[numeric_cols].var(),
        'Skewness': graduates_person_level[numeric_cols].skew(),
        'Kurtosis': graduates_person_level[numeric_cols].kurtosis()
    }).round(3)
    print(additional_stats.to_string(index=False))

print("\n" + "="*60)

In [ ]:
# Categorical variables summary
print("=== CATEGORICAL VARIABLES SUMMARY ===")
categorical_cols = graduates_person_level.select_dtypes(include=['object', 'category']).columns
print(f"Number of categorical variables: {len(categorical_cols)}")

if len(categorical_cols) > 0:
    print("\nCategorical variables overview:")
    cat_summary = []
    for col in categorical_cols:
        unique_count = graduates_person_level[col].nunique()
        most_frequent = graduates_person_level[col].mode().iloc[0] if len(graduates_person_level[col].mode()) > 0 else 'N/A'
        most_frequent_count = graduates_person_level[col].value_counts().iloc[0] if len(graduates_person_level[col].value_counts()) > 0 else 0
        
        cat_summary.append({
            'Variable': col,
            'Unique_Values': unique_count,
            'Most_Frequent': str(most_frequent)[:50] + ('...' if len(str(most_frequent)) > 50 else ''),
            'Most_Frequent_Count': most_frequent_count,
            'Most_Frequent_Pct': (most_frequent_count / len(graduates_person_level)) * 100
        })
    
    cat_df = pd.DataFrame(cat_summary)
    print(cat_df.round(2).to_string(index=False))

print("\n" + "="*60)

In [ ]:
# Boolean/Binary variables summary
print("=== BOOLEAN/BINARY VARIABLES SUMMARY ===")
bool_cols = []
binary_cols = []

# Identify boolean and binary columns
for col in graduates_person_level.columns:
    unique_vals = graduates_person_level[col].dropna().unique()
    if graduates_person_level[col].dtype == 'bool':
        bool_cols.append(col)
    elif len(unique_vals) == 2 and set(unique_vals).issubset({0, 1, True, False, 'True', 'False', 'yes', 'no', 'Yes', 'No'}):
        binary_cols.append(col)

print(f"Boolean variables: {len(bool_cols)}")
print(f"Binary variables: {len(binary_cols)}")

all_binary = bool_cols + binary_cols
if len(all_binary) > 0:
    print(f"\nSummary of {len(all_binary)} boolean/binary variables:")
    binary_summary = []
    for col in all_binary:
        value_counts = graduates_person_level[col].value_counts()
        if len(value_counts) >= 2:
            binary_summary.append({
                'Variable': col,
                'True/1_Count': value_counts.iloc[0] if value_counts.index[0] in [1, True, 'True', 'yes', 'Yes'] else value_counts.iloc[1],
                'False/0_Count': value_counts.iloc[1] if value_counts.index[0] in [1, True, 'True', 'yes', 'Yes'] else value_counts.iloc[0],
                'True_Percentage': (value_counts.iloc[0] / len(graduates_person_level)) * 100 if value_counts.index[0] in [1, True, 'True', 'yes', 'Yes'] else (value_counts.iloc[1] / len(graduates_person_level)) * 100
            })
    
    if binary_summary:
        binary_df = pd.DataFrame(binary_summary)
        print(binary_df.round(2).to_string(index=False))

print("\n" + "="*60)

In [ ]:
# Key entrepreneurship and education variables summary
print("=== KEY VARIABLES ANALYSIS ===")

# Education variables
education_vars = [col for col in graduates_person_level.columns if 'education' in col.lower() or 'degree' in col.lower() or 'school' in col.lower()]
print(f"Education-related variables: {len(education_vars)}")
if education_vars:
    print("Education variables:", education_vars[:10], "..." if len(education_vars) > 10 else "")

# Entrepreneurship variables
entrepreneur_vars = [col for col in graduates_person_level.columns if any(keyword in col.lower() for keyword in ['found', 'entrepreneur', 'startup', 'cofounder', 'owner'])]
print(f"\nEntrepreneurship-related variables: {len(entrepreneur_vars)}")
if entrepreneur_vars:
    print("Entrepreneurship variables:", entrepreneur_vars[:10], "..." if len(entrepreneur_vars) > 10 else "")

# Time-windowed variables
time_vars = [col for col in graduates_person_level.columns if any(time in col for time in ['_3_years', '_5_years', '_10_years'])]
print(f"\nTime-windowed variables: {len(time_vars)}")
if time_vars:
    print("Time-windowed variables:", time_vars[:10], "..." if len(time_vars) > 10 else "")

# Major category variables
major_vars = [col for col in graduates_person_level.columns if 'major_' in col.lower()]
print(f"\nMajor category variables: {len(major_vars)}")
if major_vars:
    print("Major variables:", major_vars[:10], "..." if len(major_vars) > 10 else "")

print("\n" + "="*60)

In [ ]:
# Sample data preview for key variables
print("=== SAMPLE DATA PREVIEW ===")

# Show a sample of key variables
key_vars = ['person_id', 'university_name', 'graduation_year'] + \
           [col for col in graduates_person_level.columns if 'major_' in col][:5] + \
           [col for col in graduates_person_level.columns if 'found' in col.lower()][:5]

# Filter to existing columns
key_vars = [col for col in key_vars if col in graduates_person_level.columns]

if key_vars:
    print(f"Sample of {len(key_vars)} key variables for first 10 observations:")
    print(graduates_person_level[key_vars].head(10).to_string(max_cols=None))
else:
    print("Key variables not found, showing first 5 columns:")
    print(graduates_person_level.iloc[:10, :5].to_string())

print("\n" + "="*60)

In [ ]:
# Data quality assessment
print("=== DATA QUALITY ASSESSMENT ===")

# Check for duplicates
duplicate_count = graduates_person_level.duplicated().sum()
print(f"Duplicate rows: {duplicate_count:,} ({(duplicate_count/len(graduates_person_level)*100):.2f}%)")

# Check for completely empty rows
empty_rows = graduates_person_level.isnull().all(axis=1).sum()
print(f"Completely empty rows: {empty_rows:,}")

# Check for rows with mostly missing data (>80% missing)
missing_threshold = 0.8
mostly_missing = (graduates_person_level.isnull().sum(axis=1) / graduates_person_level.shape[1]) > missing_threshold
print(f"Rows with >{missing_threshold*100}% missing data: {mostly_missing.sum():,}")

# Check data consistency for key ID variables
if 'person_id' in graduates_person_level.columns:
    unique_ids = graduates_person_level['person_id'].nunique()
    total_rows = len(graduates_person_level)
    print(f"Unique person IDs: {unique_ids:,} (vs {total_rows:,} total rows)")
    if unique_ids != total_rows:
        print(f"  → Multiple records per person: {total_rows - unique_ids:,} duplicate person records")

print(f"\nOverall data completeness: {((graduates_person_level.notna().sum().sum()) / (graduates_person_level.shape[0] * graduates_person_level.shape[1]) * 100):.2f}%")

print("\n" + "="*60)
print("SUMMARY STATISTICS COMPLETED")
print("="*60)

## Testing internally of the graduate file
